# Notebook 08: Project Integration

**Time:** 35 minutes  
**Prerequisites:** Notebooks 02-07 complete  
**Goal:** Apply this week's tools to your capstone project and plan your data pipeline

This notebook will:
1. Review what you learned this week
2. Ask Claude to design a domain-specific data pipeline for your project
3. Run a mini pipeline on your project's data
4. Generate a project update document

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 08")

✓ Ollama client initialized
  Available models: ['nomic-embed-text:latest', 'qwen3.5:latest', 'gemma4:e2b', 'llama3:latest', 'granite4:latest']
  Default model: granite4:latest
Setup complete -- ready for Notebook 08


---

## Part 1: Week 3 Review

This week you learned:
- **Web scraping**: trafilatura (traditional) vs Crawl4AI (modern LLM-ready)
- **OCR**: Tesseract (baseline) -> Surya (layout-aware) -> Marker/Docling (2025)
- **ASR**: Whisper, faster-whisper, model size trade-offs
- **Data cleaning**: MinHash dedup, PII removal, DataTrove/FineWeb architecture
- **TTS**: edge-tts (free cloud), Kokoro (local, highest quality)
- **Voice agents**: ASR->LLM->TTS pipeline, Pipecat framework

In [5]:
# Load previous project definition (from HW1/HW2) or create a blank
prev_project = os.path.join(outputs_dir, 'my_project_update.md')
hw2_project = os.path.join(parent_dir, '..', 'Homework2-Submission', 'outputs', 'my_project_update.md')
hw1_project = os.path.join(parent_dir, '..', 'Homework1-Submission', 'outputs', 'my_project_definition.md')

project_context = "Vacation Planner"
for path in [hw2_project, hw1_project]:
    if os.path.exists(path):
        with open(path, 'r') as f:
            project_context = f.read()
        print(f"Loaded project context from: {path}")
        print(f"Content preview: {project_context[:300]}...")
        break

if not project_context:
    print("No previous project definition found.")
    print("You'll define your project focus below.")

Loaded project context from: /home/conardd/working_space/Homework3-Submission/../Homework2-Submission/outputs/my_project_update.md
Content preview: # Project Update — Week 2

**Generated:** 2026-04-02 23:31:28
**Student Name:** [Your Name]

---

## Original Project Definition (Week 1)

# My Research Agent Project
**Created:** 2026-03-21 23:43:09


# MY RESEARCH AGENT PROJECT

## 1. PROJECT TITLE
[Your project name]

## 2. THE PROBLEM
[What rese...


---

## Part 2: Data Pipeline Strategy

In [9]:
# TODO 1: Define your project's data strategy

print("=" * 65)
print("TODO 1: Project Data Pipeline Design")
print("=" * 65)
print()

project_description = "The vacation Planer will take user's request and suggest best vacation plan."
project_domain = "Travel and Leisure"

strategy_prompt = f"""I'm building this capstone project:
{project_description}

Domain: {project_domain}

Previous context:
{project_context[:500] if project_context else 'No previous context'}

This week I learned about:
- Web scraping (trafilatura, Crawl4AI)
- OCR (Tesseract, Marker, Docling)
- ASR (faster-whisper)
- Data cleaning (MinHash dedup, PII removal, quality filtering)
- TTS (edge-tts, Kokoro)
- Voice agents (Pipecat framework)

Design a data pipeline strategy for my project:
1. What data sources should I collect from? (web, PDFs, audio?)
2. Which extraction tools are best for my domain?
3. What cleaning steps are critical for my use case?
4. Should I incorporate voice capabilities? If so, how?
5. What's the estimated data volume and processing time?

Be specific and practical."""

start = time.time()
response = client.generate(
    prompt=strategy_prompt,
    system="You are a senior ML engineer helping design a data pipeline for a capstone project.",
    max_tokens=700,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo1_reflection = """
[YOUR REFLECTION HERE]
Web sites like TripAdvisor, Lonely Planet, and some travel agency's brouchures in PDF format would be great data sources for travel recommendations. For web scraping, I would use Trafilatura for its robustness in extracting content from complex web pages, and for PDFs
I might use docling for docuements collect, and trafilatura for web crawling. 
I will also use data cleaning pipelines to pre access  data.
OCR quality might be an issue for scanned brochures, so I will need to experiment with Tesseract and Marker to see which gives better results.
- Which data sources are most relevant for your project?
- Which tools from this week will you actually use in your capstone?
- What's the most challenging data quality issue you expect to face?
"""

print()
print(todo1_reflection)

TODO 1: Project Data Pipeline Design

Response in 53.2s
Model: granite4:latest
Tokens: 373 in, 577 out
Stop reason: complete
Based on your project goal of building a Vacation Planer that takes user requests and suggests the best vacation plans, here are some recommendations for designing a data pipeline strategy:

1. Data Sources:
   - Travel websites (e.g., Expedia, Booking.com) to collect information about available flights, hotels, rental cars, etc.
   - PDF documents from travel guides, blogs, and reviews to gather insights on destinations, activities, accommodations, etc.
   - Audio files of user conversations or voice assistants to understand natural language queries and preferences.

2. Extraction Tools:
   - Web scraping libraries (e.g., BeautifulSoup, Scrapy) for extracting data from websites.
   - PDF parsing tools (e.g., PyPDF2, pdfplottter) for extracting text and relevant information from travel guides and documents.
   - Speech-to-text APIs (e.g., Google Cloud Speech-to-T

---

## Part 3: Mini Pipeline Demo

In [14]:
# TODO 2: Run a mini data pipeline for your project
#
# Use at least 2 tools from this week to collect and clean
# a small sample of data relevant to your project.

print("=" * 65)
print("TODO 2: Mini Pipeline for Your Project")
print("=" * 65)
print()

# Example: scrape papers + clean them
from src.scraping_utils import scrape_arxiv_abstracts
from src.data_pipeline import run_cleaning_pipeline

# Step 1: Collect data
my_topic = "Programming Language in AI ERA"
papers = scrape_arxiv_abstracts(topic=my_topic, max_results=5)

# Step 2: Clean the data
raw_texts = [p['abstract'] for p in papers]
pipeline_result = run_cleaning_pipeline(
    texts=raw_texts,
    target_lang="en",
    save_path=os.path.join(outputs_dir, 'my_project_data.json'),
)

print(f"\nCollected and cleaned {len(pipeline_result['cleaned_texts'])} documents for your project.")

todo2_reflection = """
[YOUR REFLECTION HERE]
I  cloolect data for Prgramming language in AI era.
No documents were removed by the pipeline. I guess the data is clean enough for my use case.
Scaling to a full dataset would involve automating the scraping process to run on a schedule, and using cloud storage and processing (e.g., AWS S3 + Lambda) to handle larger volumes of data. I would also implement monitoring to track data quality and pipeline performance.    

- What data did you collect and how did you clean it?
- Were any documents removed by the pipeline? Why?
- How would you scale this to a full dataset for your project?
"""

print()
print(todo2_reflection)

TODO 2: Mini Pipeline for Your Project

Scraping arXiv: 'Programming Language in AI ERA' (max 5 papers)

  [1] Gravitational-wave lensing beyond rays: a disordered-system approach...
      Authors: Ripalta Amoruso, Ginevra Braga, Alice Garoffolo...
      Abstract: We develop a framework to describe gravitational wave propagation through a stochastic distribution of weak gravitationa...

  [2] MM-WebAgent: A Hierarchical Multimodal Web Agent for Webpage Generation...
      Authors: Yan Li, Zezi Zeng, Yifan Yang...
      Abstract: The rapid progress of Artificial Intelligence Generated Content (AIGC) tools enables images, videos, and visualizations ...

  [3] Generalization in LLM Problem Solving: The Case of the Shortest Path...
      Authors: Yao Tong, Jiayuan Ye, Anastasia Borovykh...
      Abstract: Whether language models can systematically generalize remains actively debated. Yet empirical performance is jointly sha...

  [4] Diagnosing LLM Judge Reliability: Conformal Prediction S

---

## Part 4: Generate Project Update

In [15]:
# Generate project update document
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[Not completed]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[Not completed]'

project_update = f"""# Week 3 Project Update: Pretraining Data & Voice Agents

## Project Description

{project_description}

## Data Pipeline Strategy

{response['content'] if 'error' not in response else 'Error generating strategy'}

## Mini Pipeline Results

- Documents collected: {len(raw_texts) if 'raw_texts' in dir() else 'N/A'}
- Documents after cleaning: {len(pipeline_result['cleaned_texts']) if 'pipeline_result' in dir() else 'N/A'}

## Reflections

### Data Strategy
{_todo1}

### Pipeline Execution
{_todo2}

## Tools I Plan to Use

[STUDENT: List the specific tools from Week 3 you'll use in your capstone]

## Next Steps

[STUDENT: What will you build next week with RAG and vector databases?]
"""

update_path = os.path.join(outputs_dir, 'my_project_update.md')
with open(update_path, 'w') as f:
    f.write(project_update)

print(f"Project update saved: {update_path}")

Project update saved: ../outputs/my_project_update.md


In [ ]:
# Final reflection
full_reflection = f"""
### Data Pipeline Strategy

{_todo1}

---

### Mini Pipeline Results

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="08",
    section_title="Project Integration",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

---

## Week 3 Complete!

### Submission Checklist

- [ ] All notebooks (00-08) executed with TODOs filled in
- [ ] `outputs/homework_reflection.md` -- your main deliverable (70%)
- [ ] `outputs/my_project_update.md` -- project integration (20%)
- [ ] `outputs/arxiv_papers.json` -- scraped paper data
- [ ] `outputs/cleaned_data.json` -- cleaned pipeline output
- [ ] Audio files in outputs/ -- TTS demonstrations

### What's Next?

**Week 4:** Retrieval-Augmented Generation (RAG) -- combining LLMs with external knowledge, vector databases, and LangChain.